# TP2 - Sistemas Inteligentes: Recomendador grupal de películas y libros

**Caso de negocio:** un grupo de 5 amigos nunca se pone de acuerdo en qué ver o leer. Este sistema cruza los 5 perfiles de gustos, busca candidatos en una base de sinopsis (RAG) y devuelve UNA recomendación de película y UNA de libro, cada una justificada, en tono canchero argentino.

**Arquitectura:** `ingestion/` (TMDB + Google Books) -> `data/dataset.csv` -> `rag/` (embeddings + ChromaDB) -> `agent/` (cruza perfiles, RAG, rankea) -> `prompts/` (personalidad) -> respuesta final. Ver `docs/DISENO.md` en el repo para el detalle de cada decisión.

**Cómo correr esta notebook:**
1. Subir/clonar el repo completo a Colab (celda siguiente).
2. Cargar los secrets (`GOOGLE_API_KEY`, `TMDB_API_KEY`) en el panel de secrets de Colab (ícono de llave a la izquierda) con esos mismos nombres.
3. Correr las celdas en orden.

## 1. Setup del entorno

In [1]:
# Si estás en Colab, clonar el repo
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    !git clone https://github.com/JuanBona/EntregaTPI_IA.git repo
    %cd repo
    !pip install -q -r requirements.txt

In [2]:
# Cargar API keys: en Colab desde userdata (Secrets), en local desde .env
import os

if IN_COLAB:
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    try:
        os.environ["TMDB_API_KEY"] = userdata.get("TMDB_API_KEY")
    except Exception:
        pass  # no hace falta salvo que REGENERAR_DATASET sea True (celda de abajo)
    os.environ.setdefault("GEMINI_MODEL", "gemini-3.6-flash")
    os.environ.setdefault("CHROMA_PERSIST_DIR", "./data/chroma")
    os.environ.setdefault("DATASET_CSV_PATH", "./data/dataset.csv")
else:
    from dotenv import load_dotenv

    load_dotenv()

## 2. Ingesta de datos (`/ingestion`)

**Decisión de diseño:** si `data/dataset.csv` ya está commiteado en el repo, no hace
falta correr la ingesta de nuevo (ahorra tiempo y evita gastar la cuota de las APIs). Solo correr la celda de abajo si querés refrescar el dataset.

In [3]:
REGENERAR_DATASET = False  # poner True para volver a pegarle a TMDB/Google Books

if REGENERAR_DATASET:
    !python -m ingestion.build_dataset

import pandas as pd

df = pd.read_csv(os.environ["DATASET_CSV_PATH"])
print(f"Dataset: {len(df)} filas")
df.head()

Dataset: 276 filas


,id,tipo,titulo,sinopsis,generos,rating,anio,fuente,fuente_id,autor_director,idioma
0,016d6b3ab0b8182a,pelicula,Spider-Man: Brand New Day,Han pasado cuatro años desde los acontecimient...,Ciencia ficción|Acción|Aventura,7.865,2026,tmdb,969681,Destin Daniel Cretton,en
1,349df56f205bbd4d,pelicula,Coyote vs. Acme,Después de que todos los productos fabricados ...,Comedia|Aventura|Familia,7.700,2026,tmdb,1204680,Dave Green,en
2,03dcb62e624909d7,pelicula,La Odisea,"Tras la caída de Troya, el legendario rey de Í...",Aventura|Acción|Fantasía,7.999,2026,tmdb,1368337,Christopher Nolan,en
3,4e6788c80cb297e0,pelicula,El motín,"Después de que su jefe, un industrial multimil...",Acción|Suspense,6.400,2026,tmdb,1288445,Jean-François Richet,en
4,186dfecd032c4221,pelicula,A la carrera,"En el thriller psicológico A la carrera, Maia ...",Suspense|Acción,6.736,2026,tmdb,1386315,Kevin Macdonald,en


## 3. RAG: embeddings + ChromaDB (`/rag`)

**Decisión de diseño:** embeddings locales multilingües de HuggingFace en vez de
OpenAI, para no sumar una dependencia/costo extra al stack de Gemini. Ver
`docs/DISENO.md` para el trade-off completo.

In [4]:
from rag.vectorstore import construir_vectorstore

vectorstore = construir_vectorstore()
print(f"Vectorstore listo con {vectorstore._collection.count()} documentos.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vectorstore listo con 276 documentos.


In [5]:
# Prueba rápida del retriever
from rag.retriever import retriever

retriever("thriller psicológico con giros de guión", k=3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'titulo': 'Insidious: Fuera del más allá',
  'tipo': 'pelicula',
  'generos': 'Terror|Suspense',
  'rating': 6.483,
  'anio': 2026,
  'autor_director': 'Jacob Chase',
  'sinopsis': 'La saga de terror vuelve a sus raíces sobrenaturales. Una nueva protagonista se enfrenta a misteriosas apariciones que van mucho más allá de los fenómenos de embrujamiento clásicos. El mal encuentra nuevos caminos en la realidad. Un trío de acechadores se infiltra en un tranquilo suburbio obligando a una nueva familia a entrar en el plano astral, donde descubren una aterradora verdad: el Más Allá está sangrando en el mundo real',
  'score': 3.3762810230255127},
 {'titulo': 'Blood Covenant',
  'tipo': 'pelicula',
  'generos': 'Terror',
  'rating': 5.5,
  'anio': 2026,
  'autor_director': 'Mariano Cattaneo, Spencer Keller, Kate Trefry, Bret Miller, Javier Yañez Sanz, Hasan Can Dağlı',
  'sinopsis': 'Sigue a un prometedor escritor de terror en el momento más bajo de su vida: años sin éxito, ahogado por las d

## 4. Perfiles del grupo (`/perfiles`)

Cada integrante completó su `perfiles/<nombre>.json` (ver `perfiles/README.md` para el
esquema). Se cargan todos automáticamente.

In [6]:
from agent.perfil import cargar_todos_los_perfiles, validar_generos_contra_dataset

perfiles = cargar_todos_los_perfiles()
for p in perfiles:
    print(p.nombre)

# Chequeo rapido: generos que no existen en el dataset y por eso nunca van a matchear
validar_generos_contra_dataset(perfiles)

Bruno
Juan
Lucas
Marcos
Rafa
ojo: el género 'musical' de Bruno no existe en el dataset
ojo: el género 'musical' de Lucas no existe en el dataset
ojo: el género 'policial' de Rafa no existe en el dataset
ojo: el género 'policial' de Rafa no existe en el dataset
ojo: el género 'ensayo' de Rafa no existe en el dataset


## 5. Agente mediador (`/agent` + `/prompts`)

Cruza los 5 perfiles, busca candidatos vía RAG **por separado para películas y para
libros** (así ninguno de los dos tipos ahoga al otro en la búsqueda semántica), los
rankea por afinidad grupal y le pide a Gemini, en una sola llamada, una recomendación
de película y una de libro, con el tono de `prompts/system_prompt.py`.

In [7]:
from agent.mediador import recomendar_grupal

resultado = recomendar_grupal(perfiles)
print("Criterio de búsqueda (películas):", resultado["criterio_busqueda_peliculas"])
print("Criterio de búsqueda (libros):", resultado["criterio_busqueda_libros"])
print()
print("=== Recomendación: película ===")
print(resultado["recomendacion_peliculas"])
print()
print("=== Recomendación: libro ===")
print(resultado["recomendacion_libros"])

C:\Users\bneir\AppData\Local\Temp\ipykernel_13648\630295589.py:3: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  resultado = recomendar_grupal(perfiles)
C:\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Criterio de búsqueda (películas): terror, thriller, ciencia ficcion, accion, aventura, ciencia ficcion, ciencia ficcion, accion, thriller, comedia, drama, biografia, drama, policial, misterio, Hereditary, Get Out, Mad Max: Fury Road, Arrival, Interstellar, The Bourne Identity, The Grand Budapest Hotel, Little Miss Sunshine, Prisoners, Zodiac
Criterio de búsqueda (libros): terror, misterio, fantasia, aventura, ciencia ficcion, historia, biografia, ficcion, policial, ensayo, It, The Hobbit, Dune, Sapiens, The Silent Patient

=== Recomendación: película ===
## Pelicula

Che, armar un plan de peli con este grupo está picante: Bruno quiere terminar perturbado y mirando atrás del sillón, mientras que Marcos quiere cagarse de risa sin tener que dormir con la luz prendida. Después de repasar lo que hay en la lista, yo iría de cabeza por **"Thunderbolts*"**.

Por qué me cierra: Juan y Lucas tienen asegurada la dosis de acción, ritmo y tiros que buscan, bien al hueso y sin nada de romance para r

In [8]:
# Candidatos que consideró el agente, para ver cómo llegó a la respuesta
import pandas as pd
from IPython.display import display

print("Candidatos de película:")
display(pd.DataFrame(resultado["candidatos_rankeados_peliculas"]))
print("Candidatos de libro:")
display(pd.DataFrame(resultado["candidatos_rankeados_libros"]))

Candidatos de película:


,titulo,tipo,generos,rating,anio,autor_director,sinopsis,score,score_grupal
0,Above & Below,pelicula,Acción|Suspense|Terror|Crimen,6.200,2026,Jesse V. Johnson,Un grupo de amigos se embarca en unas vacacion...,6.502670,4.669199
1,Thunderbolts*,pelicula,Acción|Ciencia ficción|Aventura,7.299,2025,Jake Schreier,Un grupo de supervillanos poco convencional es...,7.251186,4.554544
2,La última casa,pelicula,Terror|Ciencia ficción|Suspense,6.900,2026,Louis Leterrier,Una familia encerrada en su casa misteriosamen...,7.210887,4.526734


Candidatos de libro:


,titulo,tipo,generos,rating,anio,autor_director,sinopsis,score,score_grupal
0,El Libro Negro,libro,terror|ficción|ciencia ficción|thriller,10.0,2019,"Angel David Revilla, Dross Dross","""No todo festival se desarrolla con ánimo fest...",6.844511,1.946647
1,El alienista,libro,ciencia ficción|ficción,6.0,2000,"Joaquim Maria Machado de Assis, Machado de Assis",En la literatura brasileña y en el género del ...,6.188720,0.743384
2,La promesa,libro,historia|ficción|thriller,10.0,2011,Silvina Ocampo,Una mujer se inclina sobre la baranda de un tr...,7.677052,0.696884


## 6. Casos de prueba

Tres corridas con distintos perfiles/grupos.

### Caso 1 - grupo completo (los 5 perfiles de `perfiles/`)

Corrida de `recomendar_grupal()` con los perfiles de Bruno, Juan, Lucas, Marcos y Rafa
(ver el output de las celdas de arriba).

**Película:** recomendó **"Thunderbolts\*"**, descartando *Above & Below* (le falta el
humor que pide Marcos) y *La última casa* (demasiado terror para Marcos y Rafa). Le
cierra a Juan y Lucas por la acción, a Marcos por el tono de antihéroes y a Rafa porque
evita el gore explícito. El pero: Bruno se queda sin el terror perturbador que buscaba.

**Libro:** recomendó **"El alienista"**, descartando *El Libro Negro* (Marcos y Rafa
quedaban afuera por lo turbio del contenido) y *La promesa*. Le cierra a Marcos por el
tono irónico, a Rafa por la temática de la psique sin gore ni fantasía, y a Juan y Lucas
porque es un texto corto y directo. El pero: a Bruno le sigue faltando el terror
visceral.

Al buscar y rankear por separado películas y libros, la tabla de libros trae libros
reales del dataset y el LLM justifica la recomendación de libro igual que la de película.
Además, al sumar el score semántico del retriever al ranking, el mejor candidato pasó a
ser el más cercano al pedido del grupo.

### Caso 2 - el retriever distingue una query off-topic

Probando solo `rag.retriever()`: una query clara ("aventura espacial con toques de
humor") trae candidatos con distancia 3.7-6.5, mientras que una ajena al dataset
("recetas de cocina italiana") trae 10.1-10.8. Hay margen para un umbral: si el mejor
candidato tiene distancia mayor a ~9, responder que no se encontró nada en vez de
inventar una recomendación.

### Caso 3 - una query de libro sin filtro de tipo

`rag.retriever("novela policial nordica")` sin filtro devolvió 4 películas de 5
resultados: el dataset etiqueta las películas policiales como `Crimen` y el embedding
las acerca a "policial". `agent/mediador.py` ahora llama a
`rag.retriever(query, tipo="libro")`, así que los candidatos de libro son siempre
libros. El sesgo de vocabulario del embedding sigue existiendo dentro de la búsqueda de
libros; es una limitación del modelo de embeddings.

*Nota: la API gratuita de Gemini permite ~20 requests por día. Para probar con otros
perfiles alcanza con pasarle una lista de `Perfil` a `recomendar_grupal(perfiles)`.*